# NB03 - RAG with Vector Search and LLM

This notebook turns the embeddings generated from the previous notebook into vectors, readable by `pgvector`, which can be used to perform vector search. We will then use the results of the vector search to generate a response to our queries using an LLM.

## Import dependencies

In [1]:
# General libraries
import sys
import os
import json
from tqdm.notebook import tqdm
tqdm.pandas()

# Hugging Face Transformers
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

# OpenAI API
from openai import OpenAI

# Add the project root to Python path (assuming current working directory is 'notebooks')
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Own functions
from utils import get_mean_embeddings, get_top_chunks_for_country, get_top_chunks_for_query, format_context, get_country_specific_response
from climate_policy_extractor.models import get_db_session

db_url = os.getenv('DATABASE_URL')
api_key = os.getenv('AI_API_KEY')

In [2]:
# Declare ClimateBERT tokenizer and model
model_name = 'climatebert/distilroberta-base-climate-f'
local_model_save = '../local_models/climatebert/distilroberta-base-climate-f'

# Load the models from disk
tokenizer = AutoTokenizer.from_pretrained(local_model_save)
model = AutoModel.from_pretrained(local_model_save)

Some weights of the model checkpoint at ../local_models/climatebert/distilroberta-base-climate-f were not used when initializing RobertaModel: ['lm_head.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initialized from the model checkpoint at ../local_models/climatebert/distilroberta-base-climate-f and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able t

### SELECT COUNTRY HERE

In [3]:
country = "Australia"

### Set up query and compare similarity

In [4]:
query = """
        What emissions reduction target is each country in the NDC registry aiming for by 2030?
        """

query_embedding = get_mean_embeddings(query, model, tokenizer)
len(query_embedding)

768

### Vector Similarity Search

We will only take the top 20 most similar chunks for each query. The rest of the chunks are not relevant enough to be included in the search.

In [5]:
# Connect to the database
db_session = get_db_session(db_url)

try:
    # Generate query embedding
    query = "What is the GHG emissions reduction target by 2030?"
    query_embedding = get_mean_embeddings(query, model, tokenizer)
    
    # Get top chunks for a specific country
    results = get_top_chunks_for_query(query_embedding, db_session, country, 20, 0.8)
    
finally:
    db_session.close()

In [6]:
results[0].content

'Australia also reaffirms its target to achieve net zero emissions by 2050. Both targets are economy-wide emissions reduction commitments, covering all sectors and gases included in Australia’s national inventory.'

## Set up LLM prompts

In [7]:
context = format_context(results)
context

'[Doc ID: australia_english_20220601, Page: 3, Chunk ID: 8], Content: Australia also reaffirms its target to achieve net zero emissions by 2050. Both targets are economy-wide emissions reduction commitments, covering all sectors and gases included in Australia’s national inventory.\n]\n\n[Doc ID: australia_english_20220601, Page: 11, Chunk ID: 67], Content: Australia’s updated NDC is a progression on our previous 2030 target and a significant increase in ambition, committing Australia to reduce greenhouse gas emissions by 43% below 2005 levels by 2030 — half as much again as the previous target of 26 – 28% — and achieve net zero emissions by 2050.\n\n2.1.4\n]\n\n[Doc ID: australia_english_20220601, Page: 11, Chunk ID: 68], Content: Economy-wide absolute emissions reduction targets\n\nAustralia’s 2030 and 2050 targets are economy-wide absolute emissions reduction targets.\n\n2.1.5\n\nSpecial circumstances of LDCs and SIDS\n]\n\n[Doc ID: australia_english_20220601, Page: 8, Chunk ID: 44]

In [ ]:
client = OpenAI(
    base_url=os.getenv("MODEL_ENDPOINT"),
    api_key=os.getenv("AI_API_KEY"),
)

response = get_country_specific_response(
    client,
    query,
    context,
    country)

from pprint import pprint
pprint(response)


("According to the document chunks, Australia's GHG emissions reduction target "
 'by 2030 is 43% below 2005 levels [Doc ID: australia_english_20220601, Chunk '
 'ID: 8, Page: 3]. This target is also mentioned in other chunks, including '
 '[Doc ID: australia_english_20220601, Chunk ID: 67, Page: 11], [Doc ID: '
 'australia_english_20220601, Chunk ID: 9, Page: 3], and [Doc ID: '
 'australia_english_20220601, Chunk ID: 34, Page: 7].')
